# Med Center Shuttle Peak Hours Analysis and Recommendations

This notebook will analyze the APC ridership data to assist in optimizing bus deployment across the med center express shuttle. The analysis will use BusState files from January 1, 2026 - April 17, 2026. 

### Directions

**Inbound:** Carmack 2, 3, 5A and 5B -> University Hospital, Doan Hall

**Outbound:** University Hospital, Doan Hall -> Carmack 2, 3, 5A and 5B

### Key Variables Used
- STOP_ID:      The ID of the stop (401, 403, 404, 94, 95, 401, 37)
- ARRIVAL:      Arrival time of the bus at the current stop
- DEPARTURE:    Departure time of the bus at the current stop
- BOARDINGS:    Number of boardings detected by APC at the current stop 
- ALIGHTINGS:   Number of alightings detected by APC at the current stop
- HOUR:         Hour of event
- MINUTE:       Minute of event
- DATE:         Date of event

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import importlib
import seaborn as sns
import matplotlib.pyplot as plt

# Global config for analysis and viz

# boilerplate for all viz
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F8F6',
    'axes.grid':        True,
    'grid.color':       '#E0DED8',
    'grid.linewidth':   0.6,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
})

# seaborn styles set
sns.set_style("whitegrid", {
    'axes.facecolor': '#F8F8F6',
    'grid.color':       '#E0DED8',
    'grid.linewidth':   0.6,
})

# stop id and names
STOP_NAMES = {
    37: 'Doan Hall',
    94: 'Carmack 5A',
    95: 'Carmack 5B',
    401: 'University Hospital',
    403: 'Carmack 2',
    404: 'Carmack 3'
}

# bus max capacity
# based on previous analysis, 65 is generally considered the max capacity for a cabs bus
BUS_CAPACITY = 65


---

## Loading and Validation of Data

In [2]:
# go up one level from notebook folder
sys.path.append(os.path.abspath(".."))

from modules import med_center_dashboard as mcd
importlib.reload(mcd)

data_dir = "K:/AP/TTM/Data/WMC_Dashboard/analysis_data/peak_hours"
# going to have to rework the process_mc_busstate function to take in new dir

# month and date dict
timeframe_dict = {
    # 'JAN': 2026,
    # 'FEB': 2026,
    'MAR': 2026,
    'APR': 2026
}
# set dir to save cleaned files to 
repo_dir = "K:/AP/TTM/Data/WMC_Dashboard/analysis_data/peak_hours_cleaned"

# loop through and concat to once single df
df = pd.DataFrame()
for month, year in timeframe_dict.items():
    df = pd.concat([df, mcd.process_mc_busstate(year=year, month=month, current_dir=data_dir, repo_dir=repo_dir)], ignore_index=True)

# This should only have to be run once per session. ~9 minutes to run

In [27]:
# cut down df to necessary columns
# Add time col for easier time based
mc_df = df[['DATE', 'STOP_ID', 'BOARDINGS', 'ALIGHTINGS', 'RUN_ID', 'ARRIVAL', 'DEPARTURE', 'DWELL', 'HOUR', 'MINUTE']].copy()
mc_df['TIME'] = pd.to_datetime(mc_df['DATE'].astype(str) + ' ' + mc_df['ARRIVAL'].dt.time.astype(str))

# stay within timeframe 3/1/2026 - 4/14/2026
mc_df = mc_df[(mc_df['DATE'] >= '2026-03-01') & (mc_df['DATE'] <= '2026-04-14')]
mc_df.sort_values(by='TIME', inplace=True)

# Convert floats to int
mc_df['BOARDINGS'] = mc_df['BOARDINGS'].astype(int)
mc_df['ALIGHTINGS'] = mc_df['ALIGHTINGS'].astype(int)
mc_df['RUN_ID'] = mc_df['RUN_ID'].astype(int)

# Parse date and time columns to datetime format
mc_df['DATE'] = pd.to_datetime(mc_df['DATE'])
mc_df['ARRIVAL'] = pd.to_datetime(mc_df['ARRIVAL'])
mc_df['DEPARTURE'] = pd.to_datetime(mc_df['DEPARTURE'])
mc_df['DWELL'] = pd.to_timedelta(mc_df['DWELL'], unit='s')

# Add convenience columns 
mc_df['STOP_NAME'] = mc_df['STOP_ID'].map(STOP_NAMES)
mc_df['WEEKDAY'] = mc_df['DATE'].dt.day_name() # in case weekday related patterns surface

# Parse date and time columns to datetime format
mc_df['DATE'] = pd.to_datetime(mc_df['DATE'])
mc_df['ARRIVAL'] = pd.to_datetime(mc_df['ARRIVAL'])
mc_df['DEPARTURE'] = pd.to_datetime(mc_df['DEPARTURE'])
mc_df['DWELL'] = pd.to_timedelta(mc_df['DWELL'], unit='s')

# Add convenience columns 
mc_df['STOP_NAME'] = mc_df['STOP_ID'].map(STOP_NAMES)
mc_df['WEEKDAY'] = mc_df['DATE'].dt.day_name() # in case weekday related patterns surface
mc_df['DWELL_SEC']    = mc_df['DWELL'].dt.total_seconds()

# remove any cases where date == weekend date
mc_df = mc_df[mc_df['WEEKDAY'] != 'Saturday']

In [28]:
# Shape and data check
print(f'Records : {len(mc_df)}')
print(f'Date range : {mc_df["DATE"].min().date()}  - {mc_df["DATE"].max().date()}')
print(f'Unique dates : {mc_df["DATE"].dt.date.nunique()}')
print(mc_df.dtypes)
mc_df.head(3)

Records : 52727
Date range : 2026-03-02  - 2026-04-14
Unique dates : 32
DATE           datetime64[us]
STOP_ID                 int64
BOARDINGS               int64
ALIGHTINGS              int64
RUN_ID                  int64
ARRIVAL        datetime64[us]
DEPARTURE      datetime64[us]
DWELL         timedelta64[us]
HOUR                    int32
MINUTE                  int32
TIME           datetime64[us]
STOP_NAME                 str
WEEKDAY                   str
DWELL_SEC             float64
dtype: object


,DATE,STOP_ID,BOARDINGS,ALIGHTINGS,RUN_ID,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,TIME,STOP_NAME,WEEKDAY,DWELL_SEC
16770,2026-03-02,403,0,0,1502,1900-01-01 00:22:07,1900-01-01 00:22:32,0 days 00:00:25,0,22,2026-03-02 00:22:07,Carmack 2,Monday,25.0
16771,2026-03-02,404,0,0,1502,1900-01-01 00:23:10,1900-01-01 00:23:35,0 days 00:00:25,0,23,2026-03-02 00:23:10,Carmack 3,Monday,25.0
16772,2026-03-02,94,0,0,1502,1900-01-01 00:24:19,1900-01-01 00:24:26,0 days 00:00:07,0,24,2026-03-02 00:24:19,Carmack 5A,Monday,7.0


In [29]:
# Missing data check
missing = mc_df.isnull().sum()
missing

DATE          0
STOP_ID       0
BOARDINGS     0
ALIGHTINGS    0
RUN_ID        0
ARRIVAL       0
DEPARTURE     0
DWELL         0
HOUR          0
MINUTE        0
TIME          0
STOP_NAME     0
WEEKDAY       0
DWELL_SEC     0
dtype: int64

In [30]:
# Data ranges and distributions check
mc_df.describe()

,DATE,STOP_ID,BOARDINGS,ALIGHTINGS,RUN_ID,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,TIME,DWELL_SEC
count,52727,52727.000000,52727.000000,52727.000000,52727.000000,52727,52727,52727,52727.000000,52727.000000,52727,52727.000000
mean,2026-03-22 15:47:30.600261,247.134675,3.529292,3.442790,1504.337929,1900-01-01 13:18:31.265879,1900-01-01 13:19:41.596924,0 days 00:01:10.331044,12.813758,29.208489,2026-03-23 05:06:01.866140,70.331045
min,2026-03-02 00:00:00,37.000000,0.000000,0.000000,1501.000000,1900-01-01 00:00:00,1900-01-01 00:00:06,0 days 00:00:00,0.000000,0.000000,2026-03-02 00:22:07,0.000000
25%,2026-03-11 00:00:00,94.000000,0.000000,0.000000,1502.000000,1900-01-01 07:35:31.500000,1900-01-01 07:36:48.500000,0 days 00:00:36,7.000000,14.000000,2026-03-11 18:29:05.500000,36.000000
50%,2026-03-23 00:00:00,401.000000,1.000000,1.000000,1504.000000,1900-01-01 14:13:35,1900-01-01 14:15:02,0 days 00:00:53,14.000000,29.000000,2026-03-23 11:02:10,53.000000
75%,2026-04-03 00:00:00,403.000000,5.000000,5.000000,1506.000000,1900-01-01 18:25:54.500000,1900-01-01 18:27:25.500000,0 days 00:01:27,18.000000,44.000000,2026-04-03 06:56:39.500000,87.000000
max,2026-04-14 00:00:00,404.000000,53.000000,47.000000,1514.000000,1900-01-01 23:59:58,1900-01-01 23:59:59,0 days 00:22:43,23.000000,59.000000,2026-04-14 23:57:09,1363.000000
std,NaN,165.670728,5.634727,5.326048,2.879902,NaN,NaN,0 days 00:00:54.197576,6.117385,17.492716,NaN,54.197576



---
## Analysis and Visualization of the Data
**Notes:**
- mc_df denotes the dataframe with the busstate data for the sample of 3/1/2026 - 4/14/2026
- The sample timeframe of 3/1/2026 - 4/14/2026 was chosen due to the new hospital tower opening in late February 2026
    - This should give the best possible view of staffing and demand changes to the shuttle


---

## Key Metrics Summary

In [37]:
# Aggregate to hourly totals
hourly_mc_df = (
    mc_df.groupby('HOUR')
    .agg(
        total_boardings=('BOARDINGS', 'sum'),
        total_alightings=('ALIGHTINGS', 'sum'),
        avg_dwell_sec=('DWELL_SEC', 'mean')
    )
    .reset_index()
)

# sanity check of total boardings and alightings
total_boardings = hourly_mc_df['total_boardings'].sum()
total_alightings = hourly_mc_df['total_alightings'].sum()

# total number of each within 5000
e = total_boardings - total_alightings
print(f'Total boardings: {total_boardings}')
print(f'Total alightings: {total_alightings}')
print(f'Difference: {e}')
# The total boardings and alightings should theoretically be the same
# The difference is likely due to APC data collection error
# Given the large sample size, this small difference should not significantly impact the overall analysis
# Approximately 2.5% difference.

Total boardings: 186089
Total alightings: 181528
Difference: 4561
